# 2.3 Run Kilosort 4 on a processed session

**[env: kilosort]**

Sorts every probe of the processed session made by notebook 2.1
(`<processed_session>/<probe>/continuous.dat`) with Kilosort 4, using the channel map written by
notebook 2.2 (`<processed_session>/<probe>/<session_name>_<ProbeX>.mat`). All shanks of a probe are
sorted together. Results go to `<processed_session>/<probe>/<output_dir>/` (`output_dir` from the
metadata, default `kilosort4`) and can be opened in Phy.

- **Settings**: Kilosort's own defaults (`kilosort.parameters.DEFAULT_SETTINGS`, the same values as
  in the Kilosort GUI), overridden per probe by the metadata (`<probe>.kilosort_params`). Any
  Kilosort setting can be written there by name, e.g. `Th_universal: 8`; an unknown name stops
  the notebook. Every setting that differs from the default is printed with its source.
  The other keys of `kilosort_params`:
  - `bad_channels`: excluded by the channel map of notebook 2.2; this notebook only checks that the
    `.mat` excludes exactly these channels (if not, re-run notebook 2.2).
  - `do_car`: common average reference inside Kilosort (on / off is printed per probe).
  - `output_dir`: name of the results folder.
- **Existing results are never touched**: a probe whose results folder already exists is skipped
  with a warning; the other probes are sorted. To sort a probe again, delete its results folder by
  hand.
- **`kilosort_run.json`** in the results folder records the effective settings, the inputs and the
  Kilosort version. It is written last, so a results folder without it is an unfinished run.
- **Phy**: Kilosort writes the absolute path of the data into `params.py`; it is changed to a path
  relative to the results folder (`../continuous.dat`), so the processed folder can be moved.

In [ ]:
from pathlib import Path

# ========== Specify the session ==========
session_name = "IM-1979_2026-09-03"
base_dir_output = Path(r"E:/")
processed_suffix = ""   # must match notebooks 2.1 and 2.2
# =========================================

In [ ]:
import inspect
import json
import os
from datetime import datetime
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import yaml
from kilosort import run_kilosort
from kilosort.io import load_probe
from kilosort.parameters import DEFAULT_SETTINGS

metadata_path = Path.cwd().parent / "metadata" / f"{session_name}.yaml"
print(metadata_path)
with open(metadata_path) as f:
    config = yaml.safe_load(f)

processed_session = base_dir_output / config["animal"] / "ephys" / (config["sessions"]["processed"] + processed_suffix)
probe_names = config["streams"]["probes"]
print(f"processed session: {processed_session}")
print(f"probes: {', '.join(probe_names)}")
print(f"Kilosort {version('kilosort')}")

boundaries_path = processed_session / "session_boundaries.json"
if not boundaries_path.exists():
    raise FileNotFoundError(f"{boundaries_path} not found -- run notebook 2.1 first")
streams = json.loads(boundaries_path.read_text())["streams"]

In [ ]:
# --- per probe: Kilosort defaults, overridden by <probe>.kilosort_params in the metadata ---
OWN_KEYS = {"bad_channels", "do_car", "output_dir"}         # used by this notebook / notebook 2.2
FIXED_SETTINGS = {"n_chan_bin", "fs"}                       # taken from the recording
# keyword arguments of run_kilosort that can be set from the metadata (the others are set below)
RUN_ARGUMENTS = set(inspect.signature(run_kilosort).parameters) - {
    "settings", "probe", "probe_name", "filename", "data_dir", "file_object", "results_dir", "data_dtype",
    "do_CAR", "bad_channels", "shank_idx", "device", "progress_bar", "verbose_console", "verbose_log"}
run_defaults = {name: parameter.default for name, parameter in inspect.signature(run_kilosort).parameters.items()
                if name in RUN_ARGUMENTS}

sort_config = {}
for probe_name in probe_names:
    kilosort_params = config[probe_name]["kilosort_params"]
    overrides = {key: value for key, value in kilosort_params.items() if key not in OWN_KEYS}
    unknown = sorted(set(overrides) - (set(DEFAULT_SETTINGS) - FIXED_SETTINGS) - RUN_ARGUMENTS)
    if unknown:
        raise ValueError(f"{probe_name}: unknown Kilosort setting(s) in the metadata: {unknown} "
                         "(n_chan_bin and fs are taken from the recording)")

    settings = {**DEFAULT_SETTINGS, "n_chan_bin": streams[probe_name]["n_chan"], "fs": streams[probe_name]["fs"]}
    settings.update({key: value for key, value in overrides.items() if key in DEFAULT_SETTINGS})
    run_arguments = {**run_defaults, **{key: value for key, value in overrides.items() if key in RUN_ARGUMENTS}}
    do_car = bool(kilosort_params["do_car"])
    sort_config[probe_name] = {
        "settings": settings,
        "run_arguments": run_arguments,
        "do_CAR": do_car,
        "metadata_overrides": overrides,
        "results_dir": processed_session / probe_name / kilosort_params.get("output_dir", "kilosort4"),
    }

    print(f"\n========== {probe_name}: CAR {'ON' if do_car else 'OFF'} ==========")
    print(f"  n_chan_bin = {settings['n_chan_bin']}, fs = {settings['fs']} (from the recording)")
    if overrides:
        for key, value in overrides.items():
            default = DEFAULT_SETTINGS[key] if key in DEFAULT_SETTINGS else run_defaults[key]
            print(f"  {key} = {value}   (metadata; Kilosort default {default})")
    else:
        print("  all other settings: Kilosort defaults")

In [ ]:
# --- inputs (notebooks 2.1 and 2.2) and existing results ---
probes_to_sort = []
for probe_name in probe_names:
    probe_dir = processed_session / probe_name
    n_chan = streams[probe_name]["n_chan"]
    dat_path = probe_dir / "continuous.dat"

    if not (probe_dir / ".preprocess_done").exists():
        raise FileNotFoundError(f"{probe_name}: {probe_dir / '.preprocess_done'} not found -- "
                                "notebook 2.1 did not finish this probe")
    expected_bytes = streams[probe_name]["n_samples_total"] * n_chan * 2
    if dat_path.stat().st_size != expected_bytes:
        raise ValueError(f"{probe_name}: {dat_path} has {dat_path.stat().st_size} bytes, expected {expected_bytes}")

    # channel map: must exclude exactly the bad channels of the metadata
    mat_paths = sorted(probe_dir.glob(f"{session_name}_*.mat"))
    if len(mat_paths) != 1:
        raise FileNotFoundError(f"{probe_name}: expected one {session_name}_*.mat in {probe_dir}, "
                                f"found {[p.name for p in mat_paths]} -- run notebook 2.2")
    mat = scipy.io.loadmat(mat_paths[0])
    kept = (mat["chanMap"].ravel() - 1).astype(int)
    shanks = mat["kcoords"].ravel().astype(int)
    excluded = sorted(set(range(n_chan)) - set(kept.tolist()))
    bad_channels = sorted(config[probe_name]["kilosort_params"]["bad_channels"])
    per_shank = ", ".join(f"shank {s}: {int((shanks == s).sum())}" for s in np.unique(shanks))
    print(f"{probe_name}: {mat_paths[0].name}, {len(kept)} of {n_chan} channels ({per_shank}), excluded {excluded}")
    if excluded != bad_channels:
        raise ValueError(f"{probe_name}: the channel map excludes {excluded}, the metadata bad_channels are "
                         f"{bad_channels} -- re-run notebook 2.2")
    sort_config[probe_name]["mat_path"] = mat_paths[0]
    sort_config[probe_name]["dat_path"] = dat_path

    results_dir = sort_config[probe_name]["results_dir"]
    if results_dir.exists() and any(results_dir.iterdir()):
        if (results_dir / "kilosort_run.json").exists():
            print(f"  WARNING: already sorted ({results_dir}), skipped")
        else:
            print(f"  WARNING: {results_dir} exists but the run did not finish, skipped; "
                  "delete it by hand to sort again")
        continue
    probes_to_sort.append(probe_name)

print(f"\nto sort: {probes_to_sort if probes_to_sort else 'nothing'}")

In [ ]:
# --- run Kilosort 4 per probe, then kilosort_run.json and a relative data path in params.py ---
for probe_name in probes_to_sort:
    cfg = sort_config[probe_name]
    results_dir = cfg["results_dir"]
    print(f"\n========== {probe_name}: sorting, CAR {'ON' if cfg['do_CAR'] else 'OFF'} ==========")
    print(f"  data:    {cfg['dat_path']}")
    print(f"  results: {results_dir}")

    started = datetime.now().isoformat(timespec="seconds")
    ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate, kept_spikes = run_kilosort(
        settings={**cfg["settings"], "filename": cfg["dat_path"], "results_dir": results_dir},
        probe=load_probe(cfg["mat_path"]),
        data_dtype="int16",
        do_CAR=cfg["do_CAR"],
        **cfg["run_arguments"],
    )
    print(f"{probe_name}: {int(is_ref.sum())} good units / {len(is_ref)} units")

    # Phy: data path relative to the results folder, so that the processed folder can be moved
    params_path = results_dir / "params.py"
    lines = params_path.read_text().splitlines()
    for i, line in enumerate(lines):
        if line.startswith("dat_path = "):
            data_file = results_dir / "temp_wh.dat" if cfg["run_arguments"]["save_preprocessed_copy"] else cfg["dat_path"]
            lines[i] = f"dat_path = {[Path(os.path.relpath(data_file, results_dir)).as_posix()]}"
            print(f"  params.py: {lines[i]}")
    params_path.write_text("\n".join(lines) + "\n")

    # written last: marks a finished run
    run_record = {
        "session_name": session_name,
        "probe": probe_name,
        "kilosort_version": version("kilosort"),
        "started": started,
        "finished": datetime.now().isoformat(timespec="seconds"),
        "data": {"path": str(cfg["dat_path"]), "bytes": cfg["dat_path"].stat().st_size},
        "channel_map": {"file": cfg["mat_path"].name,
                        "excluded_channels": sorted(config[probe_name]["kilosort_params"]["bad_channels"])},
        "do_CAR": cfg["do_CAR"],
        "metadata_overrides": cfg["metadata_overrides"],
        "settings": cfg["settings"],
        "run_arguments": cfg["run_arguments"],
        "n_units": int(len(is_ref)),
        "n_good_units": int(is_ref.sum()),
    }
    (results_dir / "kilosort_run.json").write_text(json.dumps(run_record, indent=2, default=str))
    print(f"  {results_dir / 'kilosort_run.json'}")

In [ ]:
# --- summary per sorted probe (also for probes sorted in an earlier run) ---
for probe_name in probe_names:
    results_dir = sort_config[probe_name]["results_dir"]
    if not (results_dir / "kilosort_run.json").exists():
        continue
    ops = np.load(results_dir / "ops.npy", allow_pickle=True).item()
    fs = float(ops["settings"]["fs"])
    amplitudes = pd.read_csv(results_dir / "cluster_Amplitude.tsv", sep="\t")["Amplitude"].values
    contam_pct = pd.read_csv(results_dir / "cluster_ContamPct.tsv", sep="\t")["ContamPct"].values
    chan_map = np.load(results_dir / "channel_map.npy")
    templates = np.load(results_dir / "templates.npy")
    best_channel = chan_map[(templates ** 2).sum(axis=1).argmax(axis=-1)]
    spike_times = np.load(results_dir / "spike_times.npy").ravel()
    spike_clusters = np.load(results_dir / "spike_clusters.npy").ravel()
    duration_s = min(ops["settings"]["tmax"], streams[probe_name]["n_samples_total"] / fs) - ops["settings"]["tmin"]
    firing_rates = np.bincount(spike_clusters, minlength=len(amplitudes)) / duration_s
    is_good = contam_pct < 10.0

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), dpi=300)
    fig.suptitle(f"{session_name} {probe_name}: {len(amplitudes)} units, {int(is_good.sum())} with contamination < 10%",
                 fontsize=10)
    if ops["dshift"] is not None:   # None when nblocks = 0 (no drift correction)
        axes[0, 0].plot(np.arange(ops["Nbatches"]) * ops["settings"]["batch_size"] / fs, ops["dshift"], lw=0.5)
    axes[0, 0].set(xlabel="time (s)", ylabel="drift (µm)", title="drift")

    first = spike_times < 5 * fs
    axes[0, 1].scatter(spike_times[first] / fs, best_channel[spike_clusters[first]], s=0.5, color="k", alpha=0.25)
    axes[0, 1].set(xlabel="time (s)", ylabel="channel", title="spikes (first 5 s)", xlim=[0, 5],
                   ylim=[chan_map.max(), 0])

    axes[0, 2].hist(firing_rates, 20, color="0.5")
    axes[0, 2].set(xlabel="firing rate (Hz)", ylabel="# units")

    axes[1, 0].hist(amplitudes, 20, color="0.5")
    axes[1, 0].set(xlabel="amplitude", ylabel="# units")

    axes[1, 1].hist(np.minimum(100, contam_pct), np.arange(0, 105, 5), color="0.5")
    axes[1, 1].axvline(10, color="k", ls="--")
    axes[1, 1].set(xlabel="% contamination", ylabel="# units", title="< 10% = good")

    axes[1, 2].scatter(firing_rates[~is_good], amplitudes[~is_good], s=3, color="r", alpha=0.25, label="mua")
    axes[1, 2].scatter(firing_rates[is_good], amplitudes[is_good], s=3, color="b", alpha=0.25, label="good")
    axes[1, 2].set(xlabel="firing rate (Hz)", ylabel="amplitude", xscale="log", yscale="log")
    axes[1, 2].legend(fontsize=8)
    fig.tight_layout()
    plt.show()